# Module 13 — Deployment and Operations
### BITS Pilani Corporate Programme | Advanced / Practitioner Level

**Audience assumption:** model training, APIs, software engineering, and basic ML concepts have already been covered.

This module treats ML deployment as a **socio-technical production system**.

## Coverage
1. Build-time vs runtime evaluation
2. Model packaging and reproducibility
3. Production inference contracts
4. FastAPI model serving
5. Request validation and schema evolution
6. Latency and throughput benchmarking
7. Observability: logs, metrics, traces
8. Data quality monitoring
9. Prediction monitoring
10. Drift detection
11. Performance monitoring with delayed labels
12. Shadow / canary deployment
13. Human feedback loops
14. Incident simulation
15. Production AI failure case studies
16. A practical MLOps reference architecture

The emphasis is on **operational failure modes**, not on explaining what an API or monitoring dashboard is.

In [ ]:
# Core imports
import time
import json
import uuid
import logging
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import Dict, Any, Optional, List

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 1. Build-time evaluation vs runtime evaluation

### Build-time
Questions include:

- Does the candidate model beat the baseline?
- Is calibration acceptable?
- Is the model within latency budget?
- Does it pass robustness tests?
- Are required artifacts reproducible?

### Runtime
Questions include:

- Are requests valid?
- Are features arriving with the expected distribution?
- Is latency within SLO?
- Are prediction distributions changing?
- Is the model still useful once labels arrive?
- Are humans overriding predictions unusually often?

A key principle:

> Passing offline evaluation is necessary but not sufficient for production validity.

In [ ]:
# A minimal build-time gate
@dataclass
class BuildGate:
    min_auc: float = 0.80
    max_brier: float = 0.20
    max_p95_latency_ms: float = 50.0

def evaluate_build_gate(metrics: Dict[str, float], gate: BuildGate):
    checks = {
        "auc": metrics["auc"] >= gate.min_auc,
        "brier": metrics["brier"] <= gate.max_brier,
        "latency": metrics["p95_latency_ms"] <= gate.max_p95_latency_ms,
    }
    return {
        "passed": all(checks.values()),
        "checks": checks,
        "metrics": metrics,
        "gate": asdict(gate)
    }

evaluate_build_gate(
    {"auc": 0.84, "brier": 0.14, "p95_latency_ms": 17},
    BuildGate()
)

## 2. Train a production-like candidate model

The model below is intentionally simple; the operational structure is the important part.

Treat:

- preprocessing,
- model,
- feature order,
- threshold,
- model version

as one deployable artifact.

In [ ]:
X, y = make_classification(
    n_samples=12000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.80, 0.20],
    class_sep=1.2,
    random_state=RANDOM_STATE
)

feature_names = [f"feature_{i}" for i in range(X.shape[1])]

split = int(len(X) * 0.8)
X_train, X_eval = X[:split], X[split:]
y_train, y_eval = y[:split], y[split:]

production_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

production_model.fit(X_train, y_train)
p_eval = production_model.predict_proba(X_eval)[:, 1]

offline_metrics = {
    "auc": roc_auc_score(y_eval, p_eval),
    "brier": brier_score_loss(y_eval, p_eval),
}

offline_metrics

## 3. Model artifact metadata

Do not deploy a model file without metadata.

At minimum capture:

- model version,
- training dataset version,
- feature schema,
- training timestamp,
- code version / git SHA,
- dependency environment,
- threshold,
- expected input ranges,
- owner,
- rollback version.

In [ ]:
model_metadata = {
    "model_name": "customer_risk",
    "model_version": "1.4.0",
    "training_dataset": "customer_training_v17",
    "feature_schema_version": "features_v3",
    "threshold": 0.50,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "owner": "risk-ml-team",
    "git_sha": "EXAMPLE_GIT_SHA",
    "rollback_version": "1.3.2",
    "features": feature_names,
    "expected_ranges": {
        f: {"min": float(np.percentile(X_train[:, i], 0.5)),
            "max": float(np.percentile(X_train[:, i], 99.5))}
        for i, f in enumerate(feature_names)
    }
}

print(json.dumps(model_metadata, indent=2))

## 4. Production inference contract

The production API should reject malformed inputs rather than silently coercing them.

Example contract:

- exactly 10 features,
- numeric values,
- no unexpected fields,
- model version returned with the prediction,
- request ID generated for traceability.

In [ ]:
def validate_request(payload: Dict[str, Any], expected_features: List[str]):
    received = set(payload.keys())
    expected = set(expected_features)

    missing = expected - received
    extra = received - expected

    if missing:
        raise ValueError(f"Missing features: {sorted(missing)}")
    if extra:
        raise ValueError(f"Unexpected features: {sorted(extra)}")

    for feature in expected_features:
        value = payload[feature]
        if not isinstance(value, (int, float, np.integer, np.floating)):
            raise TypeError(f"{feature} must be numeric")

    return True

sample_request = {f: float(X_eval[0, i]) for i, f in enumerate(feature_names)}
validate_request(sample_request, feature_names)
print("Request contract valid.")

## 5. Production prediction function

Keep the prediction boundary thin and deterministic.

Business workflows should generally not embed hidden feature transformations inside the HTTP handler.

In [ ]:
MODEL_VERSION = model_metadata["model_version"]
THRESHOLD = model_metadata["threshold"]

def predict_one(payload: Dict[str, Any]) -> Dict[str, Any]:
    request_id = str(uuid.uuid4())

    validate_request(payload, feature_names)

    row = np.array([[payload[f] for f in feature_names]], dtype=float)

    start = time.perf_counter()
    probability = float(production_model.predict_proba(row)[0, 1])
    latency_ms = (time.perf_counter() - start) * 1000

    decision = int(probability >= THRESHOLD)

    return {
        "request_id": request_id,
        "model_version": MODEL_VERSION,
        "probability": probability,
        "decision": decision,
        "latency_ms": latency_ms,
        "timestamp_utc": datetime.now(timezone.utc).isoformat()
    }

predict_one(sample_request)

## 6. FastAPI serving example

Install for the serving exercise:

```bash
pip install fastapi uvicorn pydantic
```

Save the following as `app.py` and run:

```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

The example shows the production boundary; in a real system the model artifact would be loaded at startup rather than trained in the application.

In [ ]:
# app.py example — illustrative, requires fastapi + uvicorn
fastapi_code = r'''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict
from typing import Dict

app = FastAPI(title="Customer Risk Model API", version="1.4.0")

FEATURES = [
    "feature_0", "feature_1", "feature_2", "feature_3", "feature_4",
    "feature_5", "feature_6", "feature_7", "feature_8", "feature_9"
]

class PredictionRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    feature_0: float
    feature_1: float
    feature_2: float
    feature_3: float
    feature_4: float
    feature_5: float
    feature_6: float
    feature_7: float
    feature_8: float
    feature_9: float

class PredictionResponse(BaseModel):
    request_id: str
    model_version: str
    probability: float
    decision: int
    latency_ms: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/ready")
def ready():
    # In production: verify model artifact, feature dependencies,
    # downstream services, and configuration.
    return {"ready": True}

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    payload = request.model_dump()
    try:
        result = predict_one(payload)
        return result
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))
'''

print(fastapi_code)

## 7. Latency benchmarking

Average latency is not sufficient.

Production systems care about:

- p50,
- p95,
- p99,
- throughput,
- cold-start latency,
- queueing,
- concurrency.

A system with 5 ms average latency and 500 ms p99 can still violate a customer-facing SLO.

In [ ]:
def benchmark_latency(model, X, n_warmup=100, n_runs=2000):
    # Warm-up
    for i in range(min(n_warmup, len(X))):
        model.predict_proba(X[i:i+1])

    latencies = []
    for i in range(n_runs):
        idx = i % len(X)
        start = time.perf_counter()
        model.predict_proba(X[idx:idx+1])
        latencies.append((time.perf_counter() - start) * 1000)

    latencies = np.asarray(latencies)

    return {
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "max_ms": latencies.max(),
        "mean_ms": latencies.mean()
    }

latency_report = benchmark_latency(production_model, X_eval)
latency_report

## 8. Structured logging

Do not rely on free-form application logs for model operations.

Useful fields:

- timestamp,
- request ID,
- model version,
- latency,
- feature schema version,
- prediction,
- probability,
- outcome when eventually known,
- error category.

Avoid logging sensitive raw inputs unless there is a justified and governed reason.

In [ ]:
logger = logging.getLogger("ml_service")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(handler)

def log_prediction(result: Dict[str, Any]):
    event = {
        "event": "prediction",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        **result
    }
    logger.info(json.dumps(event))

result = predict_one(sample_request)
log_prediction(result)

## 9. Operational metrics

A useful monitoring stack separates:

### System metrics
- request rate,
- error rate,
- CPU/memory,
- queue depth,
- latency percentiles.

### Data metrics
- missingness,
- range violations,
- cardinality,
- schema changes,
- feature freshness.

### Model metrics
- prediction distribution,
- class rate,
- score distribution,
- calibration,
- delayed-label accuracy.

### Business metrics
- approval rate,
- fraud capture,
- conversion,
- revenue,
- manual review rate.

The last category is often the most important.

In [ ]:
class RuntimeMetrics:
    def __init__(self):
        self.latencies_ms = []
        self.predictions = []
        self.errors = 0
        self.requests = 0

    def record(self, result):
        self.requests += 1
        self.latencies_ms.append(result["latency_ms"])
        self.predictions.append(result["probability"])

    def record_error(self):
        self.requests += 1
        self.errors += 1

    def summary(self):
        lat = np.asarray(self.latencies_ms)
        pred = np.asarray(self.predictions)

        return {
            "requests": self.requests,
            "errors": self.errors,
            "error_rate": self.errors / max(self.requests, 1),
            "p50_latency_ms": np.percentile(lat, 50) if len(lat) else np.nan,
            "p95_latency_ms": np.percentile(lat, 95) if len(lat) else np.nan,
            "p99_latency_ms": np.percentile(lat, 99) if len(lat) else np.nan,
            "mean_prediction": pred.mean() if len(pred) else np.nan,
            "prediction_std": pred.std() if len(pred) else np.nan
        }

runtime_metrics = RuntimeMetrics()

for i in range(500):
    payload = {f: float(X_eval[i % len(X_eval), j])
               for j, f in enumerate(feature_names)}
    runtime_metrics.record(predict_one(payload))

runtime_metrics.summary()

## 10. Data-quality monitoring

Monitoring should detect failures before model performance collapses.

Examples:

- missingness spike,
- impossible values,
- unexpected categorical values,
- stale features,
- unit changes,
- upstream pipeline outage.

A particularly dangerous failure is **plausible but wrong data**: values remain within valid ranges but their meaning has changed.

In [ ]:
def data_quality_report(X_current: pd.DataFrame, reference: pd.DataFrame):
    rows = []

    for col in reference.columns:
        ref = reference[col]
        cur = X_current[col]

        rows.append({
            "feature": col,
            "reference_missing_rate": ref.isna().mean(),
            "current_missing_rate": cur.isna().mean(),
            "missing_rate_delta": cur.isna().mean() - ref.isna().mean(),
            "reference_mean": ref.mean(),
            "current_mean": cur.mean(),
            "mean_delta": cur.mean() - ref.mean(),
            "reference_std": ref.std(),
            "current_std": cur.std()
        })

    return pd.DataFrame(rows)

reference_df = pd.DataFrame(X_train, columns=feature_names)
current_df = pd.DataFrame(X_eval[:1000], columns=feature_names)

data_quality_report(current_df, reference_df).head()

## 11. PSI drift detection

Population Stability Index (PSI) is one practical drift indicator.

Interpretation is context-dependent; do not blindly treat a fixed PSI threshold as a universal law.

The engineering pattern is:

1. define reference population,
2. define bins,
3. measure current distribution,
4. alert on sustained/material change,
5. investigate before retraining.

In [ ]:
def psi(reference, current, bins=10, epsilon=1e-6):
    reference = np.asarray(reference)
    current = np.asarray(current)

    cut_points = np.quantile(reference, np.linspace(0, 1, bins + 1))
    cut_points = np.unique(cut_points)

    if len(cut_points) < 3:
        return np.nan

    ref_bins = np.histogram(reference, bins=cut_points)[0] + epsilon
    cur_bins = np.histogram(current, bins=cut_points)[0] + epsilon

    ref_pct = ref_bins / ref_bins.sum()
    cur_pct = cur_bins / cur_bins.sum()

    return np.sum(
        (cur_pct - ref_pct) * np.log(cur_pct / ref_pct)
    )

drift_rows = []
for col in feature_names:
    drift_rows.append({
        "feature": col,
        "psi": psi(reference_df[col], current_df[col])
    })

pd.DataFrame(drift_rows).sort_values("psi", ascending=False)

## 12. Prediction drift

Feature drift does not always translate into prediction drift, and prediction drift can occur even when feature-level drift looks small.

Monitor:

- mean score,
- quantiles,
- positive decision rate,
- score histogram,
- calibration once labels arrive.

In [ ]:
baseline_scores = production_model.predict_proba(X_train)[:, 1]
current_scores = production_model.predict_proba(X_eval[:2000])[:, 1]

prediction_drift = {
    "baseline_mean": baseline_scores.mean(),
    "current_mean": current_scores.mean(),
    "mean_delta": current_scores.mean() - baseline_scores.mean(),
    "baseline_p95": np.percentile(baseline_scores, 95),
    "current_p95": np.percentile(current_scores, 95),
    "decision_rate_baseline": np.mean(baseline_scores >= THRESHOLD),
    "decision_rate_current": np.mean(current_scores >= THRESHOLD)
}

prediction_drift

## 13. Delayed-label monitoring

In many production systems the real outcome is not known immediately.

Examples:

- loan default: months later,
- customer churn: weeks later,
- fraud chargeback: days later,
- equipment failure: after the event.

Therefore monitoring needs an **outcome join**:

`prediction event -> entity/request ID -> eventual label`

The following code simulates delayed evaluation.

In [ ]:
# Simulate prediction events
prediction_log = pd.DataFrame({
    "request_id": [str(uuid.uuid4()) for _ in range(len(X_eval))],
    "score": p_eval,
    "prediction": (p_eval >= THRESHOLD).astype(int),
    "model_version": MODEL_VERSION
})

# Outcome arrives later
outcome_log = pd.DataFrame({
    "request_id": prediction_log["request_id"].values,
    "actual": y_eval
})

joined = prediction_log.merge(outcome_log, on="request_id", how="inner")

delayed_metrics = {
    "n_labeled": len(joined),
    "auc": roc_auc_score(joined["actual"], joined["score"]),
    "brier": brier_score_loss(joined["actual"], joined["score"]),
    "positive_rate": joined["prediction"].mean()
}

delayed_metrics

## 14. Canary deployment simulation

A canary sends a small percentage of traffic to the new model.

Typical guardrails:

- latency,
- error rate,
- prediction distribution,
- business KPI,
- model quality when labels become available.

The important point is that **traffic exposure and model approval are separate decisions**.

In [ ]:
def simulate_canary(
    champion_scores,
    candidate_scores,
    canary_fraction=0.05,
    random_state=42
):
    rng_local = np.random.default_rng(random_state)
    n = len(champion_scores)

    is_canary = rng_local.random(n) < canary_fraction

    served_score = np.where(
        is_canary,
        candidate_scores,
        champion_scores
    )

    return pd.DataFrame({
        "is_canary": is_canary,
        "served_score": served_score,
        "champion_score": champion_scores,
        "candidate_score": candidate_scores
    })

# Example candidate: a shifted threshold / slightly different model
candidate_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=0.3, max_iter=2000))
])
candidate_model.fit(X_train, y_train)

candidate_scores = candidate_model.predict_proba(X_eval)[:, 1]

canary = simulate_canary(
    p_eval, candidate_scores, canary_fraction=0.10
)

canary.groupby("is_canary").agg(
    traffic=("served_score", "size"),
    mean_score=("served_score", "mean"),
    p95_score=("served_score", lambda x: np.percentile(x, 95))
)

## 15. Shadow deployment

Shadow mode:

- production request is sent to champion;
- candidate also receives the request;
- candidate output is **not used for decisions**;
- compare outputs and operational behavior.

Shadow deployment is useful when the cost of a wrong production decision is high.

In [ ]:
shadow_df = pd.DataFrame({
    "champion": p_eval,
    "candidate": candidate_scores
})

shadow_df["absolute_difference"] = (
    shadow_df["candidate"] - shadow_df["champion"]
).abs()

shadow_df["decision_disagreement"] = (
    (shadow_df["champion"] >= THRESHOLD) !=
    (shadow_df["candidate"] >= THRESHOLD)
)

shadow_summary = {
    "mean_abs_score_difference": shadow_df["absolute_difference"].mean(),
    "p95_abs_score_difference": shadow_df["absolute_difference"].quantile(0.95),
    "decision_disagreement_rate": shadow_df["decision_disagreement"].mean()
}

shadow_summary

## 16. Human feedback loop

Human review should not be treated as an unstructured comments box.

Capture:

- model prediction,
- confidence,
- human decision,
- reason code,
- reviewer role,
- timestamp,
- eventual outcome where possible.

Then separate:

- **label correction**,
- **policy override**,
- **model error**,
- **business exception**.

This prevents every human override from being interpreted as a model-label error.

In [ ]:
feedback = pd.DataFrame({
    "request_id": prediction_log["request_id"].head(1000),
    "model_prediction": prediction_log["prediction"].head(1000),
    "model_score": prediction_log["score"].head(1000),
    "human_decision": rng.binomial(1, 0.25, 1000),
    "reason_code": rng.choice(
        ["policy_exception", "model_error", "missing_context", "customer_request"],
        1000
    )
})

feedback["override"] = (
    feedback["model_prediction"] != feedback["human_decision"]
)

feedback.groupby("reason_code").agg(
    cases=("request_id", "size"),
    overrides=("override", "sum"),
    override_rate=("override", "mean")
).sort_values("override_rate", ascending=False)

## 17. Incident simulation: upstream feature unit change

A classic production AI failure is a feature whose semantics change while its schema remains unchanged.

Example:

- training: income measured in INR,
- upstream system: starts sending income in paise,
- schema still says `income: float`.

The model remains healthy from a software perspective and may still return probabilities, but the business behavior becomes nonsensical.

In [ ]:
# Simulate a feature unit change
X_incident = X_eval.copy()

# Pretend feature_0 is an income-like feature.
# Multiply it to mimic a unit change.
X_incident[:, 0] = X_incident[:, 0] * 100

incident_scores = production_model.predict_proba(X_incident)[:, 1]

incident_report = {
    "baseline_mean_score": p_eval.mean(),
    "incident_mean_score": incident_scores.mean(),
    "baseline_positive_rate": np.mean(p_eval >= THRESHOLD),
    "incident_positive_rate": np.mean(incident_scores >= THRESHOLD),
    "score_shift": incident_scores.mean() - p_eval.mean()
}

incident_report

## 18. Incident simulation: silent model quality degradation

A dangerous failure is not a 500 error.

The API can remain:

- available,
- fast,
- syntactically correct,

while business performance deteriorates.

Simulate label shift and evaluate the consequences.

In [ ]:
# Same predictions, but altered outcome prevalence / relationship.
# This is a toy simulation for teaching purposes.
incident_y = (p_eval + rng.normal(0, 0.25, len(p_eval)) > 0.60).astype(int)

incident_quality = {
    "auc": roc_auc_score(incident_y, p_eval),
    "brier": brier_score_loss(incident_y, p_eval),
    "positive_rate_actual": incident_y.mean(),
    "positive_rate_predicted": np.mean(p_eval >= THRESHOLD)
}

incident_quality

## 19. Failure case study: data drift without immediate metric failure

A useful classroom discussion:

1. Feature distribution changes.
2. Monitoring raises a drift alert.
3. Offline performance remains unknown because labels are delayed.
4. Business team wants no interruption.
5. ML team wants retraining.
6. Product team wants to continue serving.

The correct response is not automatically "retrain."

Investigate:

- Is the drift real?
- Is the new population legitimate?
- Is the relationship between features and labels unchanged?
- Did the upstream process change?
- Is the current model outside its intended domain?
- Is there a safe fallback?

## 20. Failure case study: feedback-loop bias

Suppose a model decides who receives manual review.

Only reviewed cases receive high-quality labels.

Then retraining on reviewed outcomes creates:

`model -> selects cases -> humans label selected cases -> next model learns selection policy`

This can create self-reinforcing blind spots.

A mitigation strategy may include:

- random audit samples,
- exploration traffic,
- unbiased outcome collection,
- propensity-aware evaluation,
- separate monitoring population.

## 21. Failure case study: automation bias

Human operators may accept model recommendations more often as model confidence increases.

If the model is wrong systematically for a subgroup, human review may fail to correct it.

Therefore measure:

- override rate,
- override reason,
- reviewer disagreement by slice,
- time spent reviewing,
- outcomes after override,
- confidence vs override relationship.

Human-in-the-loop is not automatically safer than automation.

## 22. Failure case study: model/version mismatch

A common deployment failure:

- model artifact = v1.4
- feature transformation = v1.3
- threshold configuration = v1.2
- monitoring dashboard = querying v1.1 metrics

Each component can work independently while the system is logically inconsistent.

The solution is to make versions explicit and traceable in every prediction event.

In [ ]:
# Version-consistency check
deployment_manifest = {
    "model_version": "1.4.0",
    "feature_schema_version": "features_v3",
    "threshold_version": "threshold_v5",
    "monitoring_schema_version": "monitoring_v2"
}

def validate_deployment_manifest(manifest):
    required = {
        "model_version",
        "feature_schema_version",
        "threshold_version",
        "monitoring_schema_version"
    }
    missing = required - set(manifest)
    if missing:
        raise ValueError(f"Missing deployment metadata: {missing}")
    return True

validate_deployment_manifest(deployment_manifest)

## 23. Production monitoring dashboard specification

A useful executive/engineering dashboard should have four layers.

### Layer 1 — Service health
- traffic
- errors
- p50/p95/p99 latency
- saturation

### Layer 2 — Data health
- schema violations
- missingness
- feature freshness
- drift

### Layer 3 — Model health
- score distribution
- decision rate
- calibration
- delayed-label performance

### Layer 4 — Business health
- revenue / loss avoided
- conversion
- fraud captured
- manual review rate
- customer complaints
- override rate

A dashboard with only CPU and latency is not an ML monitoring dashboard.

## 24. Minimal Prometheus instrumentation example

Install:

```bash
pip install prometheus-client
```

The following demonstrates the metric design. In a real service, expose these metrics through the service's `/metrics` endpoint.

In [ ]:
prometheus_example = r'''
from prometheus_client import Counter, Histogram, Gauge

REQUESTS = Counter(
    "model_requests_total",
    "Total prediction requests",
    ["model_version", "status"]
)

LATENCY = Histogram(
    "model_request_latency_seconds",
    "Prediction latency",
    ["model_version"]
)

PREDICTION_MEAN = Gauge(
    "model_prediction_mean",
    "Rolling mean prediction score",
    ["model_version"]
)

# In request handler:
# REQUESTS.labels(MODEL_VERSION, "success").inc()
# LATENCY.labels(MODEL_VERSION).observe(latency_seconds)
# PREDICTION_MEAN.labels(MODEL_VERSION).set(score)
'''
print(prometheus_example)

## 25. Reference deployment architecture

A practical production architecture:

```text
                    ┌─────────────────────┐
                    │  CI/CD + Registry   │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │ Model / Artifact    │
                    │ Registry             │
                    └──────────┬──────────┘
                               │
                  ┌────────────▼────────────┐
                  │ Serving Layer            │
                  │ API / Batch / Stream     │
                  └────────────┬────────────┘
                               │
             ┌─────────────────┼─────────────────┐
             │                 │                 │
      ┌──────▼──────┐   ┌─────▼─────┐   ┌──────▼──────┐
      │ Logs        │   │ Metrics   │   │ Traces      │
      └──────┬──────┘   └─────┬─────┘   └──────┬──────┘
             │                │                │
             └────────────────┼────────────────┘
                              ▼
                     ┌──────────────────┐
                     │ Observability    │
                     │ + Alerting       │
                     └────────┬─────────┘
                              │
                     ┌────────▼─────────┐
                     │ Evaluation Store │
                     │ + Outcome Labels │
                     └────────┬─────────┘
                              │
                     ┌────────▼─────────┐
                     │ Retraining /     │
                     │ Human Feedback   │
                     └──────────────────┘
```

## 26. Final operational gate

A model should not be promoted merely because offline AUC increased.

A stronger promotion gate asks:

### Model
- Offline quality acceptable?
- Calibration acceptable?
- Robustness acceptable?
- Slice performance acceptable?

### Runtime
- p95/p99 within SLO?
- Error rate within budget?
- Resource consumption acceptable?

### Data
- Feature freshness verified?
- Schema verified?
- Distribution within expected domain?

### Operations
- Monitoring active?
- Alerts tested?
- Rollback tested?
- Model/version lineage recorded?

### Business
- KPI hypothesis defined?
- Human override path defined?
- Failure mode and fallback defined?

**Recommended classroom exercise:** give teams a candidate model with higher AUC but worse latency, calibration, subgroup recall, and business cost. Ask them to decide whether to deploy it and defend the decision.

# Suggested advanced lab extensions

For a full BITS corporate session, extend this notebook with:

1. **MLflow** — experiment/model registry and deployment metadata.
2. **FastAPI + Docker** — containerized serving.
3. **Prometheus + Grafana** — service/model observability.
4. **Evidently or equivalent** — data/prediction drift reports.
5. **OpenTelemetry** — distributed traces connecting API → feature service → model → downstream action.
6. **Kubernetes** — replicas, readiness/liveness probes, rolling deployment.
7. **Kafka / streaming** — event-driven prediction and delayed outcome joins.
8. **Feature store** — training/serving skew experiment.
9. **Canary controller** — automatic rollback based on SLO/business thresholds.
10. **Human feedback store** — reason-coded review workflow.